<a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-ads-recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/wl-classification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab image-classification notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> is an open-source PyTorch tool for dataset debugging, mislabel detection, and mid-training data curation. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details, and raise an issue on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a> for support.</div>

# Advertising CTR Recommendation with WeightsLab (tabular)

This notebook trains a **Wide & Deep** click-through-rate (CTR) model — the core of an advertising recommender — and instruments it with WeightsLab so every signal traces **back to the exact impression**.

A sample is one ad impression (a **row**); the model input **is** the packed field vector (8 embedded categorical fields + 8 numeric features), which WeightsLab sends to the UI as a `vector`. Each field (`ad_category`, `placement`, `bid_price`, …) is a **sortable column** in the List view.

### What you'll do
1. Install WeightsLab.
2. Generate reproducible ad impressions at a calibrated ~20% CTR.
3. Wrap the Wide & Deep model, optimizer, dataloaders, loss and metric.
4. Train while streaming per-impression signals to Weights Studio.

*Real drop-in datasets: Criteo, Avazu, MovieLens.*

## Setup

Install WeightsLab from PyPI. These tabular demos are tiny — the free Colab CPU runtime is plenty (no GPU needed).

> The **tabular input path** (the feature vector is sent to the UI as a `vector` — not a fake image) needs a WeightsLab build with tabular support (the `252-tabular-experiment` line or a release that includes it).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
%pip install torch torchvision

In [ ]:
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev6"

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as training vs. evaluation so signals are attributed to the right phase.

In [ ]:
import tempfile, logging
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torchmetrics.classification import Accuracy
from tqdm.auto import tqdm

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. A synthetic CTR dataset (rows, not images)

`make_synthetic_ctr` builds impressions with 8 categorical fields + 8 numeric features and a binary `clicked` label at ~20% CTR, driven by per-field effects plus an `ad_category x user_segment` interaction. The dataset returns the **standardized** field vector (categorical indices + per-feature standardized numerics) as the model input, and `get_items(...)` exposes the **raw**, human-readable field values (categorical labels, dollars, years, page counts, …) as metadata columns. The **Wide & Deep** model embeds each categorical field and MLPs over the embeddings + numeric features.

In [ ]:
CATEGORICAL_FIELDS = ["user_segment", "ad_category", "device_type", "os",
                      "publisher", "placement", "region", "hour_bucket"]
CARD = [6, 10, 4, 4, 12, 5, 8, 6]
NUMERIC_FIELDS = ["ad_position", "bid_price", "user_age", "session_depth",
                  "historical_ctr", "days_since_last_click", "num_ads_seen_today",
                  "creative_freshness"]
NUM_CAT, NUM_NUM = len(CATEGORICAL_FIELDS), len(NUMERIC_FIELDS)
VOCABS = {"device_type": ["mobile", "desktop", "tablet", "ctv"],
          "os": ["android", "ios", "windows", "other"],
          "placement": ["banner", "sidebar", "interstitial", "native", "video"],
          "hour_bucket": ["night", "early", "morning", "midday", "afternoon", "evening"]}
TARGET_CTR = 0.20


def _label(field, code):
    v = VOCABS.get(field)
    return v[code] if v and 0 <= code < len(v) else f"{field}_{code}"


def _draw_raw_numeric(rng, n):
    """8 numeric fields at their natural, human-readable scale (dollars, years, counts, ...)."""
    return np.stack([
        rng.integers(1, 9, n).astype(np.float64),     # ad_position: slot 1-8
        rng.gamma(2.0, 1.25, n),                       # bid_price: $, mean ~2.50
        np.clip(rng.normal(35.0, 12.0, n), 18, 80),    # user_age
        rng.poisson(3.0, n).astype(np.float64) + 1,    # session_depth (pages)
        rng.beta(2.0, 25.0, n),                        # historical_ctr, mean ~7%
        rng.exponential(10.0, n),                       # days_since_last_click
        rng.poisson(12.0, n).astype(np.float64) + 1,   # num_ads_seen_today
        rng.exponential(20.0, n),                       # creative_freshness (days)
    ], axis=1)


def make_synthetic_ctr(n, seed=0):
    """Return (cat[int], num_std[float], num_raw[float], y): shared ground-truth CTR model (seed 12345)."""
    p = np.random.default_rng(12345)
    cat_w = [p.normal(0, 0.8, c) for c in CARD]
    num_w = p.normal(0, 0.5, NUM_NUM)
    inter = p.normal(0, 0.7, (CARD[1], CARD[0]))
    rng = np.random.default_rng(seed)
    cat = np.stack([rng.integers(0, c, n) for c in CARD], axis=1).astype(np.int64)
    num_raw = _draw_raw_numeric(rng, n)
    mean, std = num_raw.mean(0, keepdims=True), num_raw.std(0, keepdims=True)
    std[std == 0] = 1.0
    num_std = (num_raw - mean) / std                  # standardized -> model input
    logit = sum(w[cat[:, f]] for f, w in enumerate(cat_w)) + num_std @ num_w
    logit = logit + inter[cat[:, 1], cat[:, 0]]
    logit -= logit.mean()
    lo, hi = -20.0, 20.0                      # calibrate bias to hit ~20% mean CTR
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if (1 / (1 + np.exp(-(logit + mid)))).mean() < TARGET_CTR:
            lo = mid
        else:
            hi = mid
    logit += 0.5 * (lo + hi)
    y = (rng.uniform(0, 1, n) < 1 / (1 + np.exp(-logit))).astype(np.int64)
    return cat, num_std.astype(np.float32), num_raw.astype(np.float32), y


class AdsCTRDataset(Dataset):
    """Yields (packed_vector, id, label); get_items() adds raw (non-standardized) field columns."""

    def __init__(self, n, seed=0):
        cat, num_std, num_raw, y = make_synthetic_ctr(n, seed)
        self.cat, self.num, self.num_raw = cat, num_std, num_raw  # self.num: standardized (model input)
        self.features = torch.from_numpy(np.concatenate([cat.astype(np.float32), num_std], 1))
        self.labels = torch.from_numpy(y)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], idx, int(self.labels[idx])

    def get_items(self, idx, include_metadata=False, include_labels=False, include_images=False):
        x = self.features[idx] if include_images else None
        target = int(self.labels[idx]) if include_labels else None
        meta = None
        if include_metadata:
            meta = {n: _label(n, int(self.cat[idx][f])) for f, n in enumerate(CATEGORICAL_FIELDS)}
            meta.update({n: round(float(self.num_raw[idx][j]), 4) for j, n in enumerate(NUMERIC_FIELDS)})
        return x, idx, target, meta


class WideDeepCTR(nn.Module):
    def __init__(self, card=CARD, num_numeric=NUM_NUM, emb=8, hidden=128, num_classes=2):
        super().__init__()
        self.card = list(card)
        self.emb = nn.ModuleList([nn.Embedding(c, emb) for c in self.card])
        self.deep = nn.Sequential(
            nn.Linear(emb * len(self.card) + num_numeric, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, num_classes))
        self.wide_cat = nn.ModuleList([nn.Embedding(c, num_classes) for c in self.card])
        self.wide_num = nn.Linear(num_numeric, num_classes)

    def forward(self, x):
        flat = x.reshape(x.shape[0], -1)
        cidx = flat[:, :len(self.card)].round().long()
        cidx = cidx.clamp(min=torch.zeros(len(self.card), device=x.device, dtype=torch.long),
                          max=torch.tensor(self.card, device=x.device) - 1)
        numeric = flat[:, len(self.card):]
        deep = self.deep(torch.cat([e(cidx[:, i]) for i, e in enumerate(self.emb)] + [numeric], 1))
        wide = self.wide_num(numeric)
        for i, e in enumerate(self.wide_cat):
            wide = wide + e(cidx[:, i])
        return deep + wide


def build_dataset(n, seed=0):
    return AdsCTRDataset(n, seed)


def build_model():
    return WideDeepCTR()

## 3. Configuration

Every tunable lives here in one dict, like a `config.yaml` with comments. Wrapping it with `flag="hyperparameters"` lets Weights Studio read (and live-edit) these values while training.

In [ ]:
config = {
    "experiment_name": "ads_ctr_wide_deep",
    "device": str(device),
    "root_log_dir": tempfile.mkdtemp(prefix="weightslab_ads_"),
    "learning_rate": 0.005,
    "training_steps_to_do": 3000,
    "eval_full_to_train_steps_ratio": 150,
    "write_export_ratio": 500,
    "class_weights": [1.0, 4.0],           # [no-click, click] — up-weight clicks
    "dataset": {"seed": 0, "n_train": 8000, "n_test": 2000},
    "data": {"train_loader": {"batch_size": 64},
             "test_loader": {"batch_size": 256}},
}
wl.watch_or_edit(config, flag="hyperparameters", poll_interval=1.0)

## 4. Wrap the training objects

This is the heart of WeightsLab. Each object passes through `wl.watch_or_edit(...)` with a `flag` describing its role. The tracked dataset's `get_items()` exposes every feature as a **sortable column**; `preload_metadata=True` loads them at init.

In [ ]:
cfg = config
train_ds = build_dataset(cfg['dataset']['n_train'], seed=cfg['dataset']['seed'])
test_ds  = build_dataset(cfg['dataset']['n_test'],  seed=cfg['dataset']['seed'] + 1)

model = wl.watch_or_edit(build_model().to(device), flag='model', device=device)
optimizer = wl.watch_or_edit(
    optim.Adam(model.parameters(), lr=cfg['learning_rate']), flag='optimizer')

train_loader = wl.watch_or_edit(
    train_ds, flag='data', loader_name='train_loader',
    batch_size=cfg['data']['train_loader']['batch_size'], shuffle=True,
    is_training=True, preload_labels=True, preload_metadata=True)
test_loader = wl.watch_or_edit(
    test_ds, flag='data', loader_name='test_loader',
    batch_size=cfg['data']['test_loader']['batch_size'], shuffle=False,
    is_training=False, preload_labels=True, preload_metadata=True)

# Class weights counter the minority (positive) class prevalence.
cw = torch.tensor(cfg['class_weights'], dtype=torch.float32, device=device)
train_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='train-loss-CE', log=True)
test_criterion = wl.watch_or_edit(
    nn.CrossEntropyLoss(weight=cw, reduction='none'),
    flag='loss', signal_name='test-loss-CE', log=True)
metric = wl.watch_or_edit(
    Accuracy(task='multiclass', num_classes=2).to(device),
    flag='metric', signal_name='metric-ACC', log=True)

## 5. Train and evaluate steps

The `guard_training_context` / `guard_testing_context` blocks tell WeightsLab the phase. `criterion(..., batch_ids=ids, preds=preds)` passes the sample ids so loss is stored **per sample**, and `wl.save_signals(...)` logs a custom per-sample accuracy signal during evaluation.

In [ ]:
def train(loader, model, optimizer, criterion, device):
    with guard_training_context:
        inputs, ids, labels = next(loader)
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(inputs)
        preds = logits.argmax(dim=1, keepdim=True)
        loss = criterion(logits.float(), labels.long(), batch_ids=ids, preds=preds)
        total = loss.mean()
        total.backward()
        optimizer.step()
    return total.detach().cpu().item()


def test(loader, model, criterion, metric, device, n_batches):
    losses = torch.tensor(0.0, device=device)
    for inputs, ids, labels in loader:
        with guard_testing_context:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            preds = logits.argmax(dim=1, keepdim=True)
            losses += criterion(logits, labels, batch_ids=ids, preds=preds).mean()
            metric.update(logits, labels)
            correct = (preds.view(-1) == labels.view(-1)).float()
            wl.save_signals(preds_raw=logits, targets=labels, batch_ids=ids,
                            signals={"test_metric/Accuracy_per_sample": correct}, preds=preds)
    return (losses / max(1, n_batches)).item(), (metric.compute() * 100).item()


## 6. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server (non-blocking) and a `bore.pub` tunnel so Weights Studio on your own machine can reach this Colab backend. `wl.start_training(...)` flips the experiment into the *training* state, then we run the loop.

In [ ]:
wl.serve(serving_grpc=True, serving_bore=True)

In [ ]:
wl.start_training(timeout=3)

steps = config['training_steps_to_do']
eval_ratio = config['eval_full_to_train_steps_ratio']
n_test_batches = len(test_loader)

test_loss = test_acc = None
pbar = tqdm(range(steps), desc='Training')
for step in pbar:
    age = model.get_age() if hasattr(model, 'get_age') else step
    train_loss = train(train_loader, model, optimizer, train_criterion, device)
    if age > 0 and age % eval_ratio == 0:
        test_loss, test_acc = test(test_loader, model, test_criterion, metric, device, n_test_batches)
    postfix = {'loss': f'{train_loss:.3f}'}
    if test_acc is not None:
        postfix['test_acc'] = f'{test_acc:.1f}%'
    pbar.set_postfix(postfix)

wl.write_history()
wl.write_dataframe()
print('Training complete.')

## See it live in Weights Studio

Everything above runs headless. The payoff is **Weights Studio**, where each row is one record and every feature is a **sortable column**.

Studio runs as a local Docker stack, and **Colab has no Docker daemon**, so you run Studio on your own machine and point it at this notebook's backend via the `bore.pub:<port>` endpoint **printed in Section 6**.

**On your machine** (with Docker Desktop):
```bash
pip install weightslab
weightslab start                     # opens http://localhost:5173
weightslab tunnel bore.pub:12345     # in another window, the host:port printed in Section 6
```

Then open **http://localhost:5173** and switch the Data Exploration board to the **List** view.

## Curate in the UI

In Weights Studio, switch the Data Exploration board to the **List** view:

1. **Sort by `test-loss-CE` descending** and generate its histogram to find the impressions the model ranks worst.
2. **Lock** the loss sort, then add `ad_category` or `placement` as a secondary sort to see which segments the model struggles on.
3. **Right-click a column** to clone, reset the sort, or generate a histogram.
4. **Discard** noisy impressions and keep training to sharpen the ranker.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>